# Pull data

In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name='dustin-payment-analysis'):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


## Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tblMin as\n'
 '(\n'
 'select\n'
 '\tintDebtorKey,\n'
 '\tmin(intLNRiskViewSKey) as intLNRiskViewSKey\n'
 'from DimLNRiskView\n'
 'group by intDebtorKey\n'
 ')\n'
 ' \n'
 'select\n'
 '\tDimLNRiskView.intAccountKey as bigAccountId,\n'
 '\ttblMin.intDebtorKey as bigDebtorId,\n'
 '\tDimLNRiskView.dtmStampCreation,\n'
 '\tDimLNRiskView.intScore\n'
 'from tblMin\n'
 'left outer join DimLNRiskView on '
 'DimLNRiskView.intLNRiskViewSKey=tblMin.intLNRiskViewSKey')


## Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=edw;'
    'Database=pfsedw;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# info
print(f'Data contains {df.shape[0]} rows and {df.shape[1]} columns')

Data contains 4536454 rows and 4 columns
Wall time: 2min 42s


In [7]:
# preview
df.head()

,bigAccountId,bigDebtorId,dtmStampCreation,intScore
0,2930717,3791492,2017-01-02 13:25:05.3333333,568
1,2930720,3791497,2017-01-02 13:25:18.2033333,621
2,2903942,3757380,2016-12-14 09:44:22.7733333,597
3,2887273,3736221,2016-12-02 15:24:54.2033333,599
4,2915723,3772390,2016-12-22 09:49:23.4400000,519


In [8]:
for col in df.columns:
    print(col)

bigAccountId
bigDebtorId
dtmStampCreation
intScore


In [9]:
# get prop nan
df.isnull().mean()

bigAccountId        0.0
bigDebtorId         0.0
dtmStampCreation    0.0
intScore            0.0
dtype: float64

### Rename columns

In [10]:
%%time

# drop
df.drop('dtmStampCreation', axis=1, inplace=True)

# rename
dict_rename = {
    'bigAccountId': 'bigaccountid__app',
    'bigDebtorId': 'bigdebtorid__app',
    'intScore': 'intscore__ln'
}
df.rename(columns=dict_rename, inplace=True)

# show
df

Wall time: 60 ms


,bigaccountid__app,bigdebtorid__app,intscore__ln
0,2930717,3791492,568
1,2930720,3791497,621
2,2903942,3757380,597
3,2887273,3736221,599
4,2915723,3772390,519
...,...,...,...
4536449,7304311,9084742,222
4536450,7304557,9085036,566
4536451,7302932,9083081,529
4536452,7305058,9085655,598


### Save

In [11]:
%%time

# save
str_filename = 'df_intscore.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 23.4 s


## Upload to s3

In [12]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/get_intscore/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 23.7 s


## Clean-up

In [13]:
os.remove(str_local_path)